# ViT

# ConvNeXt
# ConvNeXt-Tiny Optimization Strategy: Phase 

* without metadata(NDVI, Height etc)
* composite loss mae:rmse = 65:35
* moduler architecture


## Configuration
### Main model
* unfreeze layer - 15
* LR - 1e-4
* WD - 1e-2
* Loss - 0.65

### FInetune
* unfreeze layer - stage_3
* LR - 1e-5
* WD - 1e-4
* Loss - 0.65

## Main

### Libraries

In [ ]:
!pip install vit-keras tensorflow-addons

### imports

In [ ]:
import os
import time
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import random
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from IPython.display import FileLink, display

import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

In [ ]:
# static seed
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

def set_deterministic_environment(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"✅ Environment locked to Seed: {seed}")

set_deterministic_environment(SEED)

In [ ]:
PATH = '/kaggle/input/csiro-biomass'
IMG_DIR = PATH 
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 500
K = 5
UNFREEZE_LAYERS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
DROP_OUT = 0.3
ALPHA = 0.65
LR_SCHEDULER_MONITOR = "val_loss"
LR_SCHEDULER_PATIENCE = 10
LR_SCHEDULER_FACTOR = 0.1
EARLY_STOP_MONITOR = "val_loss"
EARLY_STOP_PATIENCE = 15

FT_LEARNING_RATE = 1e-5
FT_WEIGHT_DECAY = 1e-4
FT_EPOCHS = 100
FT_ALPHA = 0.65
FT_LR_SCHEDULER_MONITOR = "val_loss"
FT_LR_SCHEDULER_PATIENCE = 10
FT_LR_SCHEDULER_FACTOR = 0.6
FT_EARLY_STOP_MONITOR = "val_loss"
FT_EARLY_STOP_PATIENCE = 5

TRAIN_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
FINAL_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']

### composite loss function

In [ ]:
def stabilized_mae_mse_loss(alpha=0.65, target_weights=None, epsilon=1e-6):
    """
    Composite loss:
    alpha * MAE + (1 - alpha) * RMSE
    Optional per-target weighting
    """
    if target_weights is not None:
        # Ensure weights are a float32 tensor for multiplication
        target_weights = tf.constant(target_weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        error = y_true - y_pred

        # Per-element MAE and squared error
        mae = tf.abs(error)
        sq = tf.square(error)

        # Apply per-target weights BEFORE reducing (error (batch, 3) * weights (3,))
        if target_weights is not None:
            mae = mae * target_weights
            sq = sq * target_weights

        # True RMSE: take the MEAN of the squared error FIRST, then sqrt.
        # (The old version did sqrt(sq + eps) per element, which is just |error| == MAE,
        #  so the RMSE term contributed nothing. Reducing before the sqrt makes the
        #  loss actually penalize large errors more heavily.)
        rmse = tf.sqrt(tf.reduce_mean(sq) + epsilon)

        return alpha * tf.reduce_mean(mae) + (1 - alpha) * rmse

    return loss

# def log_cosh_loss(target_weights=None, rmse_weight=0.5):
#     """
#     Log-Cosh Loss: Smooth, twice-differentiable alternative to MAE/RMSE.
#     log(cosh(x)) is approximately (x^2)/2 for small x and abs(x) - log(2) for large x.
#     """
#     if target_weights is not None:
#         target_weights = tf.constant(target_weights, dtype=tf.float32)

#     def loss(y_true, y_pred):
#         error = y_pred - y_true
        
#         # Log-cosh calculation
#         # We use the identity log(cosh(x)) = x + log(1 + exp(-2x)) - log(2) 
#         # or tf.math.log(tf.math.cosh(x)) for stability
#         loss_val = tf.math.log(tf.math.cosh(error))

#         if target_weights is not None:
#             loss_val = loss_val * target_weights

#         return tf.reduce_mean(loss_val)

#     return loss

# # def hybrid_log_cosh_rmse_loss(target_weights=None, rmse_weight=0.5):
#     """
#     Hybrid Loss: Combines Log-Cosh for stability and RMSE to maximize R2.
#     - Log-Cosh: Smoothly handles smaller errors and reduces outlier impact.
#     - RMSE: Heavily penalizes large residuals, which is critical for R2 scores.
#     """
#     if target_weights is not None:
#         target_weights = tf.constant(target_weights, dtype=tf.float32)

#     def loss(y_true, y_pred):
#         error = y_pred - y_true
        
#         # 1. Log-Cosh component (Stability)
#         # Using tf.math.log(tf.math.cosh(x)) is standard, 
#         # but for very large errors, x - log(2) is more numerically stable.
#         log_cosh = tf.math.log(tf.math.cosh(error + 1e-12))

#         # 2. RMSE component (R2 Optimizer)
#         mse = tf.math.square(error)
#         rmse = tf.math.sqrt(tf.reduce_mean(mse, axis=0) + 1e-12)

#         # Apply target weights to the base pixel-wise loss if provided
#         if target_weights is not None:
#             log_cosh = log_cosh * target_weights
#             # Weighting the RMSE per target before the final mean
#             rmse = rmse * target_weights

#         # Combine losses
#         # Adjust rmse_weight (alpha) to balance between stability and R2 score
#         total_loss = tf.reduce_mean(log_cosh) + (rmse_weight * tf.reduce_mean(rmse))

#         return total_loss

#     return loss

In [ ]:
def log_transform_targets(df, target_cols):
    df = df.copy()
    df[target_cols] = np.log1p(df[target_cols])
    return df


def inverse_log_transform_targets(arr):
    """
    Use this ONLY for evaluation / reporting
    """
    return np.expm1(arr)

### data loader

In [ ]:
df = pd.read_csv(os.path.join(PATH, 'train.csv'))
df_pivot = df.pivot_table(
    index=['image_path'], 
    columns='target_name', values='target').reset_index().fillna(0)

df_final = log_transform_targets(df_pivot, TRAIN_TARGETS)

trainval_df, test_df = train_test_split(df_final, test_size=0.20, random_state=SEED)
train_df, val_df = train_test_split(trainval_df, test_size=0.20, random_state=SEED)

# train_order_file = 'training_order_fixed.csv'
# train_df[['image_path']].to_csv(train_order_file, index=False)
# display(FileLink(train_order_file))

# ====================================================
# DATA PIPELINE
# ====================================================
def preprocess_fn(img_path, label):
    file_path = tf.strings.join([IMG_DIR, '/', img_path])
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    
    return img, label

def create_ds(df, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        df["image_path"].values.astype(str),
        df[TRAIN_TARGETS].values.astype(np.float32)
    ))
    
    if shuffle: 
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    
    return ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = create_ds(train_df), create_ds(val_df, False), create_ds(test_df, False)

### main tracker

In [ ]:
def mae_per_target(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred), axis=0).tolist()

def mse_per_target(y_true, y_pred):
    return np.mean((y_true - y_pred)**2, axis=0).tolist()

def rmse_per_target(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2, axis=0)).tolist()

def r2_per_target(y_true, y_pred):
    return r2_score(y_true, y_pred, multioutput='raw_values').tolist()

### score calculator

In [ ]:
def final_score_calculator(y_true, y_pred, ft = ""):
    """
    Computes the weighted R2 over all (image, target) pairs combined.
    Expects y_true and y_pred to be shape (N, 5) in order:
    [Green, Dead, Clover, GDM, Total]
    """
    # 1. Define weights per target type
    # Weights: Green (0.1), Dead (0.1), Clover (0.1), GDM (0.2), Total (0.5)
    weights_per_target = np.array([0.1, 0.1, 0.1, 0.2, 0.5])
    
    # 2. Flatten the arrays to treat all (image, target) pairs together
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # 3. Create a matching weights vector
    # Repeat the weights for every row (image) in the dataset
    w_flat = np.tile(weights_per_target, len(y_true))
    
    # 4. Calculate weighted mean of true values (y_bar_w)
    # Formula: sum(w_j * y_j) / sum(w_j)
    y_bar_w = np.sum(w_flat * y_true_flat) / np.sum(w_flat)
    
    # 5. Residual Sum of Squares (SS_res)
    # Formula: sum(w_j * (y_j - y_hat_j)^2)
    ss_res = np.sum(w_flat * (y_true_flat - y_pred_flat)**2)
    
    # 6. Total Sum of Squares (SS_tot)
    # Formula: sum(w_j * (y_j - y_bar_w)^2)
    ss_tot = np.sum(w_flat * (y_true_flat - y_bar_w)**2)
    
    # 7. Final R2 calculation
    # Formula: 1 - (SS_res / SS_tot)
    r2_weighted = 1 - (ss_res / ss_tot)

    print("#" * 20)
    print(f'# R2 Result {ft} : {r2_weighted}')
    print("#" * 20)
    
    return r2_weighted

### visualizer

In [ ]:
# ====================================================
# 7. METRIC CURVE DASHBOARD (UPDATED)
# ====================================================
def visualizer(model, history, dataset, df, name=""):
    # 1. Metric Plotting
    plt.figure(figsize=(18, 5))
    metrics_to_plot = [m for m in ['loss', 'mae', 'mse', 'rmse'] if m in history.history]
    
    for i, metric in enumerate(metrics_to_plot):
        plt.subplot(1, len(metrics_to_plot), i+1)
        plt.plot(history.history[metric], label=f'Train {metric}')
        plt.plot(history.history[f'val_{metric}'], label=f'Val {metric}')
        plt.title(metric.upper())
        plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
    
    # 2. Predictions
    y_pred_log = model.predict(dataset)
    y_true_log = np.concatenate([y for x, y in dataset], axis=0)
    
    # Inverse log transform and clip negatives
    y_pred_raw = np.maximum(np.expm1(y_pred_log), 0.0)
    y_true_raw = np.expm1(y_true_log)
    
    # 3. Reconstruction (5 columns)
    def reconstruct_full(data):
        g, d, c = data[:, 0], data[:, 1], data[:, 2]
        # Columns: Green, Dead, Clover, GDM (G+D), Total (G+D+C)
        return np.column_stack([g, d, c, (g + d), (g + d + c)])
    
    y_pred = reconstruct_full(y_pred_raw)
    y_true = reconstruct_full(y_true_raw)

    # files name
    target_metrics = f"target_metrics_{name}.csv"
    detailed_error = f"detailed_errors_{name}.csv"
    
    # REPORT NAMES (5 columns)
    REPORT_NAMES = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    
    # 4. Error Export
    error_export = pd.DataFrame({'image_path': df['image_path'].values})
    for i, target in enumerate(REPORT_NAMES):
        error_export[f'true_{target}'] = y_true[:, i]
        error_export[f'pred_{target}'] = y_pred[:, i]
        error_export[f'error_{target}'] = np.abs(y_true[:, i] - y_pred[:, i])
    error_export.to_csv(detailed_error, index=False)
    
    # 5. Metrics DataFrame
    metrics_df = pd.DataFrame({
        'Target': REPORT_NAMES, 
        'MAE': mae_per_target(y_true, y_pred), 
        'MSE': mse_per_target(y_true, y_pred),
        'RMSE': rmse_per_target(y_true, y_pred),
        'R2': r2_per_target(y_true, y_pred)
    })
    metrics_df.to_csv(target_metrics, index=False)
    
    print("\n--- FINAL TEST SUMMARY ---")
    print(metrics_df.to_string(index=False))
    display(FileLink(detailed_error))
    display(FileLink(target_metrics))

    # ====================================================
    # NEW EXTENSIONS ADDED BELOW
    # ====================================================
    
    # --- 1. MIN, MAX, MEDIAN, MEAN TABLES ---
    actual_stats = []
    pred_stats = []
    
    for i, target in enumerate(REPORT_NAMES):
        actual_stats.append({
            'Target': target,
            'Min': np.min(y_true[:, i]),
            'Max': np.max(y_true[:, i]),
            'Median': np.median(y_true[:, i]),
            'Mean': np.mean(y_true[:, i])
        })
        pred_stats.append({
            'Target': target,
            'Min': np.min(y_pred[:, i]),
            'Max': np.max(y_pred[:, i]),
            'Median': np.median(y_pred[:, i]),
            'Mean': np.mean(y_pred[:, i])
        })
        
    df_actual_stats = pd.DataFrame(actual_stats)
    df_pred_stats = pd.DataFrame(pred_stats)
    
    print("\n📊 --- ACTUAL VALUE STATISTICS ---")
    print(df_actual_stats.to_string(index=False))
    
    print("\n🤖 --- PREDICTED VALUE STATISTICS ---")
    print(df_pred_stats.to_string(index=False))
    print("-" * 50)

    # --- 2. PER-TARGET VISUALIZATIONS LOOP (VALUE-WISE DISTRIBUTION) ---
    for i, target in enumerate(REPORT_NAMES):
        
        # 2a. Line graph: Sorted value comparison (helps see prediction trends across value ranges)
        # We sort indices based on the actual true values ascending
        sorted_val_indices = np.argsort(y_true[:, i])
        
        plt.figure(figsize=(15, 4))
        plt.plot(range(len(df)), y_true[sorted_val_indices, i], label='Actual Value', color='teal', alpha=0.8, linewidth=2)
        plt.plot(range(len(df)), y_pred[sorted_val_indices, i], label='Predicted Value', color='crimson', linestyle='--', alpha=0.8, linewidth=1.5)
        plt.title(f"Actual vs Predicted Trend (Sorted by True Value): {target}", fontsize=12, fontweight='bold')
        plt.xlabel("Samples (Ordered from Lowest to Highest Value)")
        plt.ylabel("Value (g)")
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show()

        # 2b. Bar Graphs for Value Range Frequency Distributions (Histograms)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))
        
        # Determine consistent value bins for both plots based on minimum and maximum range
        min_val = min(np.min(y_true[:, i]), np.min(y_pred[:, i]))
        max_val = max(np.max(y_true[:, i]), np.max(y_pred[:, i]))
        bins = np.linspace(min_val, max_val, num=20) # 20 numerical buckets from lowest to highest value
        
        # Actual value range frequency bar layout
        ax1.hist(y_true[:, i], bins=bins, color='darkgreen', alpha=0.7, rwidth=0.85, edgecolor='black', linewidth=0.5)
        ax1.set_title(f"Actual Value Range Distribution: {target}", fontsize=11, color='darkgreen')
        ax1.set_xlabel("Value Ranges (g)")
        ax1.set_ylabel("Count of Images (Frequency)")
        ax1.grid(True, linestyle=':', alpha=0.5)

        # Predicted value range frequency bar layout
        ax2.hist(y_pred[:, i], bins=bins, color='darkorange', alpha=0.7, rwidth=0.85, edgecolor='black', linewidth=0.5)
        ax2.set_title(f"Predicted Value Range Distribution: {target}", fontsize=11, color='darkred')
        ax2.set_xlabel("Value Ranges (g)")
        ax2.set_ylabel("Count of Images (Frequency)")
        ax2.grid(True, linestyle=':', alpha=0.5)

        plt.tight_layout()
        plt.show()
        print("=" * 80)

    return final_score_calculator(y_true, y_pred, name)

### shap

### lime

### main model

In [ ]:
# # ====================================================
# # 5. MODEL ARCHITECTURE
# # ====================================================
# def build_model():
#     img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
    
#     # 1. DATA AUGMENTATION PIPELINE
#     aug = layers.RandomFlip("horizontal_and_vertical")(img_in) 
#     aug = layers.RandomRotation(0.2, fill_mode='reflect')(aug)  
#     aug = layers.RandomTranslation(0.1, 0.1, fill_mode='reflect')(aug)
#     aug = layers.RandomZoom(0.2)(aug)
    
#     # Color/Brightness
#     aug = layers.RandomBrightness(factor=0.2)(aug)   
#     aug = layers.RandomContrast(factor=0.2)(aug)     
    
#     # Custom Lambda Layers
#     aug = layers.Lambda(lambda x: tf.image.random_hue(x, 0.02))(aug)
#     aug = layers.Lambda(lambda x: tf.image.random_saturation(x, 0.7, 1.3))(aug)
    
#     # Noise for Regularization
#     aug = layers.GaussianNoise(0.3)(aug) 
    
#     # 2. BASE MODEL
#     # Ensure the indentation here matches the start of the function!
#     base_model = tf.keras.applications.ConvNeXtTiny(
#         include_top=False, 
#         weights='imagenet', 
#         input_tensor=aug, 
#         pooling='avg'
#     )
    
#     # 3. FINE-TUNING STRATEGY
#     base_model.trainable = True
#     # Freeze all layers except the last N
#     for layer in base_model.layers[:-UNFREEZE_LAYERS]:
#         layer.trainable = False
        
#     # 4. TOP HEAD
#     vis_out = base_model.output
#     x = layers.Dense(128, activation='silu')(vis_out)
#     x = layers.Dropout(DROP_OUT)(x) 
    
#     outputs = layers.Dense(
#         len(TRAIN_TARGETS), 
#         activation="linear",
#         name="biomass_output"
#     )(x)
    
#     return models.Model(inputs=[img_in], outputs=outputs)

# # Instantiate and Compile
# model = build_model()

# model.compile(
#     optimizer=tf.keras.optimizers.AdamW(
#         learning_rate=LEARNING_RATE, 
#         weight_decay=WEIGHT_DECAY
#     ),
#     loss=stabilized_mae_mse_loss(alpha=ALPHA),
#     metrics=[
#         tf.keras.metrics.MeanAbsoluteError(name="mae"),
#         tf.keras.metrics.MeanSquaredError(name="mse"),
#         tf.keras.metrics.RootMeanSquaredError(name="rmse")
#     ]
# )

import tensorflow as tf
from tensorflow.keras import layers, models

# ====================================================
# 5. UPDATED MODEL ARCHITECTURE (The Spatial Fix)
# ====================================================
def build_model():
    img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
    
    # 1. TUNED DATA AUGMENTATION PIPELINE
    # Removed GaussianNoise(0.3) to preserve leaf textures
    aug = layers.RandomFlip("horizontal_and_vertical")(img_in) 
    aug = layers.RandomRotation(0.2, fill_mode='reflect')(aug)  
    aug = layers.RandomTranslation(0.1, 0.1, fill_mode='reflect')(aug)
    aug = layers.RandomZoom(0.2)(aug)
    
    # Color/Brightness
    aug = layers.RandomBrightness(factor=0.2)(aug)   
    aug = layers.RandomContrast(factor=0.2)(aug)     
    
    # Custom Lambda Layers - subtle hue to keep botanical identity
    aug = layers.Lambda(lambda x: tf.image.random_hue(x, 0.02))(aug)
    aug = layers.Lambda(lambda x: tf.image.random_saturation(x, 0.8, 1.2))(aug)
    
    # CoarseDropout (Cutout) - Better for "counting" biomass than Gaussian noise
    # It forces the model to learn from partial plant views
    aug = layers.Dropout(0.1)(aug) # Subtle pixel dropout
    
    # Reduced Gaussian Noise for regularization without blurring features
    aug = layers.GaussianNoise(0.05)(aug) 
    
    # 2. BASE MODEL (Structural Change: pooling=None)
    base_model = tf.keras.applications.ConvNeXtTiny(
        include_top=False, 
        weights='imagenet', 
        input_tensor=aug, 
        pooling=None  # We keep the 2D spatial features here
    )
    
    # 3. FINE-TUNING STRATEGY
    base_model.trainable = True
    for layer in base_model.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
        
    # 4. SPATIAL ATTENTION HEAD (The Fix for Random Patterns)
    # base_model.output shape is (Batch, H, W, 768)
    spatial_features = base_model.output
    
    # # Create an attention map: learns which pixels are 'biomass' vs 'soil'
    # attention_map = layers.Conv2D(1, (1, 1), activation='sigmoid', name='spatial_attention')(spatial_features)
    
    # # Multiply features by the attention map
    # weighted_features = layers.Multiply()([spatial_features, attention_map])
    
    # # Now pool the weighted features
    # vis_out = layers.GlobalAveragePooling2D()(weighted_features)
    # Spatial + Channel
    channels = spatial_features.shape[-1]

    # --- STEP A: CHANNEL ATTENTION ---
    # Global average and max pooling across spatial dimensions to get channel descriptors
    avg_pool = layers.GlobalAveragePooling2D()(spatial_features)
    max_pool = layers.GlobalMaxPooling2D()(spatial_features)
    
    # Shared MLP to determine channel importance
    shared_layer_one = layers.Dense(channels // 16, activation='relu', kernel_initializer='he_normal', use_bias=False)
    shared_layer_two = layers.Dense(channels, kernel_initializer='he_normal', use_bias=False)
    
    avg_out = shared_layer_two(shared_layer_one(avg_pool))
    max_out = shared_layer_two(shared_layer_one(max_pool))
    
    channel_attention = layers.Add()([avg_out, max_out])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_attention = layers.Reshape((1, 1, channels))(channel_attention)
    
    # Apply Channel Attention
    channel_weighted = layers.Multiply()([spatial_features, channel_attention])
    
    # --- STEP B: SPATIAL ATTENTION (Your original logic, applied on top) ---
    spatial_attention = layers.Conv2D(1, (1, 1), activation='sigmoid', name='spatial_attention')(channel_weighted)
    weighted_features = layers.Multiply()([channel_weighted, spatial_attention])
    
    # Pool the fully weighted features
    vis_out = layers.GlobalAveragePooling2D()(weighted_features)
    
    # Top Head
    x = layers.Dense(128, activation='silu')(vis_out)
    x = layers.Dropout(DROP_OUT)(x) 
    
    outputs = layers.Dense(
        len(TRAIN_TARGETS), 
        activation="linear",
        name="biomass_output"
    )(x)
    
    return models.Model(inputs=[img_in], outputs=outputs)

# ====================================================
# COMPILATION
# ====================================================
model = build_model()

# Using the recommended Log-Cosh or your updated strategy
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE, 
        weight_decay=WEIGHT_DECAY
    ),
    loss=stabilized_mae_mse_loss(), # Refer to previous step for log_cosh code
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
        tf.keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

### K FOLD

In [ ]:
# kf = KFold(n_splits=K, shuffle=True, random_state=SEED)

# # To store results for all folds
# all_fold_metrics = {
#     'mae': [],
#     'mse': [],
#     'rmse': []
# }

# # 2. Loop through each fold
# for fold, (train_idx, val_idx) in enumerate(kf.split(df_final)):
#     print(f"\n--- TRAINING FOLD {fold + 1}/{K} ---")
    
#     # Create fold-specific DataFrames
#     train_df = df_final.iloc[train_idx]
#     val_df = df_final.iloc[val_idx]
    
#     # Create tf.data.Datasets using your create_ds function
#     train_ds = create_ds(train_df, shuffle=True)
#     val_ds = create_ds(val_df, shuffle=False)
    
#     # Re-initialize a fresh model for every fold!
#     model = build_model()
#     model.compile(
#         optimizer=tf.keras.optimizers.AdamW(
#             learning_rate=LEARNING_RATE, 
#             weight_decay=WEIGHT_DECAY
#         ),
#         loss=stabilized_mae_mse_loss(
#             alpha=ALPHA,
#         ),
#         metrics=[
#             tf.keras.metrics.MeanAbsoluteError(name="mae"),
#             tf.keras.metrics.MeanSquaredError(name="mse"),
#             tf.keras.metrics.RootMeanSquaredError(name="rmse")
#         ]
#     )

#     callbacks = [
#         tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
#         tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.6, patience=10, verbose=1),
#     ]
    
#     # Train
#     history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

#     # visualizer(model, history)
#     visualizer(
#         model,
#         history,
#         val_ds,
#         val_df,
#         name=f"fold_{fold+1}"
#     )
    
#     # Evaluate the fold
#     results = model.evaluate(val_ds, return_dict=True)
    
#     # Store each metric (ensure the keys match your model.compile names)
#     all_fold_metrics['mae'].append(results['mae'])
#     all_fold_metrics['mse'].append(results['mse'])
#     all_fold_metrics['rmse'].append(results['rmse'])
    
#     print(f"Fold {fold+1} Results: MAE: {results['mae']:.4f}, MSE: {results['mse']:.4f}, RMSE: {results['rmse']:.4f}")


# # --- FINAL AGGREGATED SUMMARY ---
# print("\n" + "="*30)
# print("FINAL CROSS-VALIDATION SUMMARY")
# print("="*30)
# for metric, values in all_fold_metrics.items():
#     mean_val = np.mean(values)
#     std_val = np.std(values)
#     print(f"Average {metric.upper()}: {mean_val:.4f} (+/- {std_val:.4f})")

In [ ]:
def get_main_lr_callback(epochs=50):
    lr_start = 1e-6
    lr_max = LEARNING_RATE  # align peak LR with the configured LEARNING_RATE (was hard-coded 5e-5)
    lr_min = 1e-6
    lr_ramp_ep = 5 # Warmup for 5 epochs

    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        else:
            # Clamp progress to [0, 1] so the cosine still fully decays even if the
            # actual run is shorter/longer than `epochs`.
            progress = (epoch - lr_ramp_ep) / max(1, (epochs - lr_ramp_ep))
            progress = min(max(progress, 0.0), 1.0)
            lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + tf.math.cos(3.14159 * progress))
        return float(lr)
    return tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=1)

### main model train

In [ ]:
# ====================================================
# 6. TRAINING
# ====================================================
print("\n--- Training Stability-Optimized ConvNext ---")

# Horizon for the cosine schedule. EPOCHS (500) is only an upper bound; EarlyStopping
# stops far sooner, so decaying over 500 epochs kept the LR pinned near its peak.
# Use a realistic horizon so the cosine actually anneals during the run.
LR_SCHEDULE_EPOCHS = 50

callbacks = [
        get_main_lr_callback(epochs=LR_SCHEDULE_EPOCHS),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

visualizer(model, history, test_ds, test_df, name="main")

### main model store in keras

In [ ]:
# Save model
model.save("pasture_biomass_model.keras")

In [ ]:
# # ====================================================
# # 1. TOTAL DETERMINISM SETUP
# # ====================================================
# def set_all_seeds(seed):
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # Operational for some TF versions
#     random.seed(seed)
#     np.random.seed(seed)
#     tf.keras.utils.set_random_seed(seed)
#     tf.config.experimental.enable_op_determinism() # The "Magic" switch

# # ====================================================
# # 2. UPDATED DATA PIPELINE
# # ====================================================
# def preprocess_fn(img_path, label):
#     file_path = tf.strings.join([IMG_DIR, '/', img_path])
#     img = tf.io.read_file(file_path)
#     img = tf.image.decode_jpeg(img, channels=3)
#     img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
#     img = tf.cast(img, tf.float32) / 255.0
#     return img, label

# def create_deterministic_ds(df, seed_val, is_training=True):
#     ds = tf.data.Dataset.from_tensor_slices((
#         df["image_path"].values.astype(str),
#         df[TRAIN_TARGETS].values.astype(np.float32)
#     ))
    
#     if is_training:
#         # Crucial: seed here matches the global loop seed
#         ds = ds.shuffle(len(df), seed=seed_val, reshuffle_each_iteration=True)
    
#     # We keep AUTOTUNE, but enable_op_determinism() handles the ordering
#     ds = ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
#     return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# # ====================================================
# # 3. OPTIMIZATION LOOP
# # ====================================================
# seeds = [42, 123, 500, 789, 1024, 2048, 5555, 7777, 8888, 9999]
# best_r2 = -float('inf')
# best_seed = None

# for attempt, seed in enumerate(seeds, 1):
#     print(f"\n{'='*30}\nATTEMPT {attempt}: Testing Seed {seed}\n{'='*30}")
    
#     # --- Step A: Reset everything to this specific seed ---
#     set_all_seeds(seed)
#     tf.keras.backend.clear_session()
    
#     # --- Step B: Prepare Data ---
#     train_ds_fixed = create_deterministic_ds(train_df, seed, is_training=True)
#     val_ds_fixed = create_deterministic_ds(val_df, seed, is_training=False)
#     test_ds_fixed = create_deterministic_ds(test_df, seed, is_training=False)
    
#     # --- Step C: Build Model (Weights now initialized by seed) ---
#     model = build_model() 
#     model.compile(
#         optimizer=tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
#         loss=stabilized_mae_mse_loss(alpha=ALPHA),
#         metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae"), "mse"]
#     )

#     callbacks = [
#         tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
#         tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.6, patience=10, verbose=0)
#     ]

#     # --- Step D: Train ---
#     history = model.fit(
#         train_ds_fixed, 
#         validation_data=val_ds_fixed, 
#         epochs=EPOCHS, 
#         callbacks=callbacks,
#         verbose=1
#     )
    
#     # --- Step E: Evaluate ---
#     # (Assuming your visualizer saves metrics to a CSV)
#     attempt_name = f"seed_{seed}"
#     current_r2 = visualizer(model, history, test_ds_fixed, test_df, name=attempt_name)
    
#     # metrics_df = pd.read_csv(f"target_metrics_{attempt_name}.csv")
#     # current_r2 = metrics_df.loc[metrics_df['Target'] == 'Dry_Total_g', 'R2'].values[0]
    
#     print(f"Seed {seed} -> R2: {current_r2:.4f}")
    
#     # --- Step F: Save Best ---
#     if current_r2 > best_r2:
#         best_r2 = current_r2
#         best_seed = seed
#         model.save('best_model.keras')
#         print(f"⭐ New Best Found! R2: {best_r2:.4f}")

#     # Cleanup memory
#     del model
#     import gc
#     gc.collect()

# print(f"\n✅ Optimization Complete. Best Seed: {best_seed} with R2: {best_r2:.4f}")

### finetune model

In [ ]:
# # ===============================
# # 6. FINE-TUNE LAST BLOCK - FIXED
# # ===============================

# # 1. Start by setting the "fine-tune" toggle
# # We want to unfreeze everything from stage_3 onwards
# fine_tune_at_layer = "stage_3_block_0" 
# keep_frozen = True

# print(f"Searching for {fine_tune_at_layer} to begin unfreezing...")

# for layer in model.layers:
#     # If we hit the target layer, stop freezing everything from here onwards
#     if fine_tune_at_layer in layer.name:
#         keep_frozen = False
    
#     if keep_frozen:
#         layer.trainable = False
#     else:
#         # This will unfreeze the stage 3 blocks AND your custom Dense/Output layers
#         layer.trainable = True

# # 2. Verify
# trainable_layers = [l.name for l in model.layers if l.trainable]
# print(f"\nTotal trainable layers: {len(trainable_layers)}")
# print("Last 5 trainable layers:", trainable_layers[-5:])

# # 3. Recompile (Important: always recompile after changing 'trainable' flags)
# model.compile(
#     optimizer=tf.keras.optimizers.AdamW(
#         learning_rate=5e-5, # Low LR for fine-tuning
#         weight_decay=WEIGHT_DECAY
#     ),
#     loss=stabilized_mae_mse_loss(alpha=ALPHA),
#     metrics=[
#         tf.keras.metrics.MeanAbsoluteError(name="mae"),
#         tf.keras.metrics.MeanSquaredError(name="mse"),
#         tf.keras.metrics.RootMeanSquaredError(name="rmse")
#     ]
# )

In [ ]:
# ====================================================
# 6. PRECISION FINE-TUNE (STAGE 3 ONLY)
# ====================================================

# 1. First, freeze EVERY layer in the entire model
model.trainable = True 
for layer in model.layers:
    layer.trainable = False

# 2. Specifically unfreeze only Stage 3 and the Head
# We use a simple name check on the full model list.
# NOTE: added "attention" so the `spatial_attention` Conv2D (the spatial-attention
# map, which HAS trainable weights) is unfrozen too. Previously only the channel-
# attention Dense layers matched ("dense"), leaving the spatial-attention conv frozen.
print("--- Confirmed Unfrozen Layers ---")
for layer in model.layers:
    # Logic: Unfreeze if it's Stage 3 OR if it's our custom top layers
    if "stage_3" in layer.name or any(x in layer.name for x in ["dense", "biomass", "dropout", "pooling", "attention"]):
        layer.trainable = True
        print(f"✅ {layer.name}")

# 3. Recompile with the proper Learning Rate

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=FT_LEARNING_RATE, 
        weight_decay=FT_WEIGHT_DECAY
    ),
    loss=stabilized_mae_mse_loss(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
        tf.keras.metrics.MeanSquaredError(name="mse"),
        tf.keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

print("\nSuccess: Only Stage 3 and the Prediction Head (incl. spatial attention) are now trainable.")

### finetune model train

In [ ]:
# 6.5 Fine-tune
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor=FT_LR_SCHEDULER_MONITOR,
    factor=FT_LR_SCHEDULER_FACTOR,
    patience=FT_LR_SCHEDULER_PATIENCE,
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor=FT_EARLY_STOP_MONITOR, 
    patience=FT_EARLY_STOP_PATIENCE, 
    restore_best_weights=True
)

historyft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    callbacks=[early_stop, lr_scheduler]  # lr_scheduler was previously built but never passed
)

visualizer(model, historyft, test_ds, test_df, name="ft")

### finetune model store in keras

In [ ]:
# Save model
model.save("pasture_biomass_model_ft.keras")

In [ ]:
def run_research_explanation(df_to_use, idx=0, save_prefix="research_fig"):
    base_output_dir = "/kaggle/working/explanations"
    id_folder = os.path.join(base_output_dir, str(idx))
    os.makedirs(id_folder, exist_ok=True)

    img_name = df_to_use.iloc[idx]['image_path']
    full_path = os.path.join(IMG_DIR, img_name)
    
    # Preprocessing
    img_raw = tf.io.read_file(full_path)
    img_tensor = tf.image.decode_jpeg(img_raw, channels=3)
    img_tensor = tf.image.resize(img_tensor, [IMG_SIZE, IMG_SIZE])
    img_normalized = (tf.cast(img_tensor, tf.float32) / 255.0).numpy()
    
    input_batch = np.expand_dims(img_normalized, axis=0)
    full_prediction = model.predict(input_batch, verbose=0).flatten()
    
    for target_idx, target_name in enumerate(TRAIN_TARGETS):
        actual_val = df_to_use.iloc[idx][target_name]
        pred_val = full_prediction[target_idx]
        
        # --- FIXED LIME ---
        explainer = lime_image.LimeImageExplainer()
        explanation = explainer.explain_instance(
            img_normalized.astype('double'), 
            # This returns a shape of (batch, 1)
            lambda x: model.predict(x, verbose=0, batch_size=10)[:, target_idx:target_idx+1], 
            labels=(0,),          # <--- CRITICAL FIX: Tell LIME to only look at column 0
            top_labels=None, 
            num_samples=500
        )
        
        # Now explanation.local_exp will only contain key 0
        temp, mask = explanation.get_image_and_mask(
            0,                     # <--- Matches the label (0,) above
            positive_only=True, 
            num_features=5, 
            hide_rest=False
        )

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        ax1.imshow(img_normalized)
        ax1.set_title(f"Original: {target_name}")
        
        # FIX: Added color=(1, 0, 0) parameter to make boundaries red
        ax2.imshow(mark_boundaries(temp, mask, color=(1, 0, 0)))
        ax2.set_title(f"LIME: Actual {actual_val:.2f} | Pred {pred_val:.2f}")
        plt.savefig(os.path.join(id_folder, f"{save_prefix}_lime_{target_name}.png"))
        plt.show()

        # --- FIXED SHAP ---
        try:
            # Using 'blur' instead of 'inpaint' is much faster for testing
            masker = shap.maskers.Image("blur(32,32)", shape=img_normalized.shape)
            
            # Explain the target specific prediction
            explainer_shap = shap.Explainer(lambda x: model.predict(x, verbose=0)[:, target_idx:target_idx+1], masker)
            shap_values = explainer_shap(input_batch, max_evals=200, batch_size=10)

            # image_plot expects (1, H, W, C) for the SHAP values
            shap.image_plot(shap_values, input_batch, show=False)
            
            plt.savefig(os.path.join(id_folder, f"{save_prefix}_shap_{target_name}.png"))
            plt.show()
            plt.close()
        except Exception as e:
            print(f"⚠️ SHAP failed for {target_name}: {e}")

In [ ]:
# ====================================================
# PERFORMANCE INDEX EXTRACTOR (COMPUTES OVERALL MAE)
# ====================================================
def get_performance_indices(df, model, target_cols):
    print("🔄 Calculating predictions for the entire dataset...")
    
    images = []
    for img_name in df['image_path']:
        full_path = os.path.join(IMG_DIR, img_name)
        
        # Match your exact preprocessing pipeline
        img_raw = tf.io.read_file(full_path)
        img_tensor = tf.image.decode_jpeg(img_raw, channels=3)
        img_tensor = tf.image.resize(img_tensor, [IMG_SIZE, IMG_SIZE])
        img_normalized = (tf.cast(img_tensor, tf.float32) / 255.0).numpy()
        images.append(img_normalized)
    
    X_test = np.array(images)
    
    # Predict across the whole batch
    preds = model.predict(X_test, batch_size=32, verbose=1)
    actuals = df[target_cols].values
    
    # Calculate Mean Absolute Error across all 3 targets simultaneously per row
    errors = np.mean(np.abs(actuals - preds), axis=1)
    
    # Store errors alongside original indices
    performance_df = pd.DataFrame({
        'original_idx': range(len(df)),
        'error': errors
    })
    
    # Sort: Smallest average error first
    performance_df = performance_df.sort_values(by='error').reset_index(drop=True)
    
    best_5_indices = performance_df.head(5)['original_idx'].tolist()
    worst_5_indices = performance_df.tail(5)['original_idx'].tolist()
    
    return best_5_indices, worst_5_indices


# ====================================================
# EXECUTION PIPELINE
# ====================================================
# Execute the search across all 3 targets
best_indices, worst_indices = get_performance_indices(test_df, model, TRAIN_TARGETS)

# --- Run for Top 5 Best Predictions ---
print("\n" + "!"*40 + "\nRUNNING EXPLANATIONS FOR BEST PREDICTIONS\n" + "!"*40)
for rank, idx in enumerate(best_indices):
    print(f"\n💎 Processing Best Image #{rank+1} (Dataset Row Index: {idx})")
    run_research_explanation(test_df, idx=idx, save_prefix=f"BEST_rank_{rank+1}")

# --- Run for Top 5 Worst Predictions ---
print("\n" + "!"*40 + "\nRUNNING EXPLANATIONS FOR WORST PREDICTIONS\n" + "!"*40)
for rank, idx in enumerate(worst_indices):
    print(f"\n❌ Processing Worst Image #{rank+1} (Dataset Row Index: {idx})")
    run_research_explanation(test_df, idx=idx, save_prefix=f"WORST_rank_{rank+1}")